# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 9.30e-14 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.00e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405022e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 4.76837e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797539388882393·1 - 0.09754482372919099·y - 0.4175945425725998·x + 2.1931519300021916·y^2 - 0.1269608752480836·x*y - 0.8025868707701475·x^2)^2 + (0.9043585861943056·1 - 0.6454975493868688·y + 0.2742173500241793·x - 0.10770166105288279·y^2 - 1.2293131412594325·x*y - 0.9300170435472996·x^2)^2 + (-0.5739156167209173·1 - 1.3853512632170424·y - 0.6219655290929101·x - 0.20409240982196922·y^2 - 0.15182435477475986·x*y + 0.4443814867923305·x^2)^2 + (0.5703017871356986·1 - 0.5184102408490359·y + 0.34367412466167085·x + 0.2555951335437905·y^2 + 0.7618448416641994·x*y - 0.02090567085116562·x^2)^2 + (0.04140298207812761·1 + 0.04911089940832097·y - 0.49990900774527314·x - 0.24792082949867686·y^2 + 0.2972448439664639·x*y - 0.5054176772471584·x^2)^2 + (-0.25245328219549756·1 - 0.06336616977802403·y + 0.25394133757830056·x - 0.10018803519680765·y^2 + 0.05960215579665097·x*y - 0.1938133953768724·x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Indeed, in that case, SumOfSquares computes the corresponding Gröbner basis which
may take a non-negligible amount of time for large systems of equalities.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  169.436 μs (6177 allocations: 1.34 MiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  285.240 μs (10996 allocations: 1.25 MiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
We can use this system in a Sum-of-Squares constraint as follows:

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 1.44e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 1.79e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.64e-01 Pobj:  0.0000000e+00 Ad: 8.07e-01 Dobj:  2.6104524e+00 
Iter:  2 Ap: 7.71e-01 Pobj:  0.0000000e+00 Ad: 7.40e-01 Dobj: -5.9092654e-02 
Iter:  3 Ap: 8.08e-01 Pobj:  0.0000000e+00 Ad: 7.65e-01 Dobj:  5.5549062e-01 
Iter:  4 Ap: 7.56e-01 Pobj:  0.0000000e+00 Ad: 7.79e-01 Dobj:  4.8506840e-01 
Iter:  5 Ap: 6.95e-01 Pobj:  0.0000000e+00 Ad: 6.77e-01 Dobj:  4.3029639e-01 
Iter:  6 Ap: 6.35e-01 Pobj:  0.0000000e+00 Ad: 6.83e-01 Dobj:  2.2137577e-01 
Iter:  7 Ap: 6.21e-01 Pobj:  0.0000000e+00 Ad: 6.43e-01 Dobj:  1.2986420e-01 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 2.60997e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-5.119321889701442e-5·1 + 0.0010563677688278982·y + 0.00034023099954502885·x + 8.152795057221345·y^2 - 32.75660820499037·x*y + 29.418081966700065·x^2)^2 + (-1.0585339651028275e-5·1 + 0.00010570284827584334·y + 0.0010714522180101445·x + 26.08236419857713·y^2 - 16.083691442712897·x*y - 25.137306718756314·x^2)^2 + (2.9691570405089302e-5·1 + 0.0007926510213909337·y - 0.00016788257670553004·x - 0.12259248180780073·y^2 - 0.09192631196464744·x*y - 0.06838391387956044·x^2)^2 + (-5.172117862769082e-7·1 - 0.037195725506381234·y + 0.04977578305675424·x - 0.0001654868304237845·y^2 - 0.00012340336123786824·x*y - 9.078560153900308e-5·x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

3.6146519377175047848577797162360203397923896773136220872402191162109375e-09 + 7.851034468053025972458465636947708072731972769804251117584032890124346108558749e-08y + 6.002425817924334366510335642557877760667067307692307692307692307692307692307756e-08x - 8.919389744236299913632097513538354860429535619914531707763671875e-06y² - 1.3337185879687208091520615738101440683749387972056865692138671875e-05xy - 4.9789874580980467385954608972031820712800254113972187042236328125e-06x² - 6.58375025608393116272054612636566162109375e-08y³ - 7.070378682527689306880347430706024169921875e-08xy² - 5.828971500329199319852431504518364135947194881737232208251953125e-08x²y + 4.956833576076090908967531644381009615384615384615384615384615384615384615384695e-09y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

4.2821112055645653504930088950782007817252861059387214481830596923828125e-09 + 1.289321759181939750188543748078446472577641730029613543779422075324146211632181e-09y + 2.366877634056681227565814669315631573016826923076923076923076923077265866382688e-09x + 4.28210668833306495884016840136609971523284912109375e-09y² + 7.3795136668053373796283267438411712646484375e-15xy + 4.2821061856969377945603127955109812319278717041015625e-09x² + 9.76996261670137755572795867919921875e-15y³ - 9.858780458671390078961849212646484375e-14xy² + 1.2538581284360361678409390151500701904296875e-13x²y + 5.127302616668972544945203340970552884615384615384615384615384615381873068939011e-09y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*